# Methods 03 - Group tasks: Bayesian answers to three climate questions

**How this works.** Your group does one task (about 20-30 minutes). All the code is already written: you only change the
lines marked `# EDIT ME`, run the cells, look at the figures and discuss the questions.

1. Run the **Setup** cell once (a few seconds). It loads the three datasets and the helper functions.
2. Jump to your task. Each task works on its own after Setup.
3. Change one thing at a time, re-run the cells of your task, and note what changes.

**What every group reports back** (two minutes at the end):

1. **How much did your prior matter?** Show the same result with two different priors.
2. **One result stated as a probability** ("the probability that ... is ...%"), not as a single number.
3. **The assumptions you made** (noise model, measurement errors, model shape), in one or two sentences.

All three tasks use the same machinery as the main notebook: a model (likelihood), priors, and the Metropolis sampler
from `bayes_helpers.py`. Every run prints its acceptance rate and shows the trace plots - check them before you
believe anything. Aim for roughly 20-40% acceptance - a rule of thumb, not a law; the trace plots are the real check.

## Setup (run once, do not edit)

What this cell does, in words:
* imports the libraries and the helpers (`bayes_helpers.py`, which packages the sampler and the two models of the main notebook);
* loads **HadCRUT5** (annual global temperature anomaly 1850-2025, with its published uncertainty range turned into a 1-sigma error per year);
* loads **NGRIP** (Greenland ice-core d18O, 20-year means, 9,540-122,280 years before 2000 CE on the GICC05 timescale);
* loads the **September Arctic sea-ice area** series 1901-2024 (reconstruction + adjusted observations, with an *assumed* 1-sigma error per era).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import bayes_helpers as bh

plt.rcParams.update({"font.size": 11, "figure.dpi": 100})

hadcrut = bh.load_hadcrut5()      # columns: year, global_anomaly, ..., tau
ngrip   = bh.load_ngrip()         # columns: age_b2k, d18O_permil
seaice  = bh.load_seaice()        # columns: year, sia_sep_nh, ..., tau_sia, source

def standardise_label(ax):
    ax.grid(True, alpha=0.4)

print("HadCRUT5:", hadcrut.year.min(), "-", hadcrut.year.max(), "|",
      "NGRIP:", len(ngrip), "values |", "sea ice:", seaice.year.min(), "-", seaice.year.max())

---
## Task 1 - HadCRUT5: how fast is the world warming?

**Goal.** Fit a straight line to the global temperature anomaly with the Bayesian machinery, and report the warming
rate as a **posterior distribution** - not a single number.
**Why.** "The world is warming at 0.2 degrees C per decade" sounds precise, but how sure are we, and how does the answer depend
on the period we look at, on the measurement errors, and on our prior? Bayesian fitting answers all three at once.
**You will get** the fit with a 95% credibility band, the posterior of the rate, and probabilities such as P(rate > 0).

The model (from `bayes_helpers.trend_model`): anomaly = a + b x (year - midpoint)/10 + noise, so **b is the rate in degrees C per decade**.
The measurement error of each year (`tau`, from the published HadCRUT5 uncertainty range) is added to the model noise - exactly
the "variances add" trick from the main notebook, except that here the errors are published, not assumed.

**Step 1.** Choose your knobs.

In [ ]:
PERIOD = (1970, 2025)              # EDIT ME: (first year, last year), inside 1850-2025. Try (1850, 2025), (1998, 2012)
PRIOR_SLOPE = None                 # EDIT ME: None = flat prior on the rate, or (mean, sd) in degrees C/decade,
                                   #          e.g. (0.0, 0.05) = "a sceptic: probably no warming"
USE_MEASUREMENT_ERROR = True       # EDIT ME: False ignores the published per-year uncertainties

**Step 2.** Run the fit (no edits needed).

In [ ]:
y_first, y_last = PERIOD
if not (1850 <= y_first < y_last <= 2025) or y_last - y_first + 1 < 15:
    raise ValueError("PERIOD must lie inside 1850-2025 and span at least 15 years.")
if PRIOR_SLOPE is not None and not (isinstance(PRIOR_SLOPE, (tuple, list))
                                    and len(PRIOR_SLOPE) == 2 and PRIOR_SLOPE[1] > 0):
    raise ValueError("PRIOR_SLOPE must be None or (mean, sd) with sd > 0, e.g. (0.0, 0.05).")

d = hadcrut[(hadcrut.year >= y_first) & (hadcrut.year <= y_last)]
tau = d["tau"].values if USE_MEASUREMENT_ERROR else None
log_post, info = bh.trend_model(d.year.values, d.global_anomaly.values, tau=tau, slope_prior=PRIOR_SLOPE)

# step sizes scaled with the amount of data: more years -> tighter posterior -> smaller steps
steps = np.array([0.012, 0.02, 0.012]) * np.sqrt(56 / len(d))
samples, acc = bh.run_mcmc(log_post, initial=[0.0, 0.0, 0.3], step_sizes=steps)
print(f"Acceptance rate: {acc:.0%}  (healthy: roughly 20-40%)")
bh.plot_traces(samples, ["a (level, °C)", "b (rate, °C/decade)", "sigma (noise, °C)"], acc)

rate = samples[:, 1]
lo, hi = np.percentile(rate, [2.5, 97.5])
ols = np.polyfit((d.year - info["x0"]) / 10, d.global_anomaly, 1)[0]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax = axes[0]
years_fine = np.linspace(y_first, y_last, 200)
pred = np.array([info["predict"](s, years_fine) for s in samples[::20]])
ax.errorbar(d.year, d.global_anomaly, yerr=2 * d["tau"], fmt="o", ms=3, lw=0.8, alpha=0.6, label="HadCRUT5 (±2σ)")
ax.plot(years_fine, pred.mean(axis=0), color="crimson", lw=2, label="mean posterior fit")
ax.fill_between(years_fine, *np.percentile(pred, [2.5, 97.5], axis=0), color="gray", alpha=0.35, label="95% credibility band")
ax.set_xlabel("Year"); ax.set_ylabel("Anomaly (°C vs 1961-1990)")
ax.set_title(f"Global temperature, {y_first}-{y_last}"); ax.legend(fontsize=9); standardise_label(ax)

ax = axes[1]
ax.hist(rate, bins=50, density=True, alpha=0.75)
ax.axvline(0, color="k", lw=1)
ax.axvline(ols, color="red", ls="--", label=f"frequentist slope ({ols:.3f})")
if PRIOR_SLOPE is not None:
    g = np.linspace(rate.min() - 0.1, rate.max() + 0.1, 300)
    ax.plot(g, stats.norm.pdf(g, *PRIOR_SLOPE), color="gray", label=f"prior N{PRIOR_SLOPE}")
ax.set_xlabel("Warming rate (°C per decade)"); ax.set_ylabel("Posterior density")
ax.set_title("Posterior of the rate"); ax.legend(fontsize=9); standardise_label(ax)
plt.tight_layout(); plt.show()

def as_prob(p, n):
    # a probability estimated from n samples is only resolved to about 1/n
    return "> 0.999 (no posterior sample below)" if p == 1 else ("< 0.001 (no posterior sample above)" if p == 0 else f"{p:.4f}")

print(f"Posterior mean rate:        {rate.mean():.3f} °C/decade   95% CI [{lo:.3f}, {hi:.3f}]")
print(f"P(rate > 0):                {as_prob(np.mean(rate > 0), len(rate))}")
print(f"P(rate > 0.2 °C/decade):    {as_prob(np.mean(rate > 0.2), len(rate))}")

**Discuss.**
1. Shorten the period (e.g. `(1998, 2012)`, the so-called "hiatus"). What happens to the width of the credible interval and to P(rate > 0.2)? Why should headlines based on short periods worry you?
2. Set `PRIOR_SLOPE = (0.0, 0.05)` - a sceptic who believes there is probably no warming. With `PERIOD = (1970, 2025)`, does the data win? And with a very short period? What does that say about when priors matter?
3. Set `USE_MEASUREMENT_ERROR = False` and re-run. Which number changes most, and why does it change so little here? (Hint: compare the size of `tau` with the model noise sigma. When would the measurement errors matter more - 1850 or 2020?)
4. With a flat prior (`PRIOR_SLOPE = None`), the posterior mean is nearly the frequentist slope (red dashed line). Why?

---
## Task 2 - NGRIP: when did the Holocene begin?

**Goal.** Fit a **step-change model** to the Greenland ice-core d18O record and get the timing of an abrupt climate
transition as a **posterior distribution** - "the warming happened at 11,7xx +- xx years b2k", with honest uncertainty.
**Why.** Abrupt climate changes are dated by eye surprisingly often. A Bayesian change-point model makes the dating
reproducible and gives the uncertainty for free. The Holocene onset also has an independently dated reference age
(11,703 years b2k, from the ice-core timescale), so you can check your posterior against it.
**You will get** the record with the fitted step, and the posterior of the change-point age T.

**The model** (from `bayes_helpers.step_model`). The record is treated as one constant level before the event, another
constant level after it, and noise:

$$
y(t) \;=\; \begin{cases} \mu_{\text{before}} & \text{if } t > T \\ \mu_{\text{after}} & \text{if } t \le T \end{cases}
\;\; + \;\epsilon, \qquad \epsilon \sim \mathcal{N}(0, \sigma^2)
$$

Four unknowns: the two levels, the change-point age $T$, and the noise $\sigma$. **The intuition:** pick a candidate $T$
and the model splits the record in two; the likelihood is high when each half sits flat around its own mean, and low when
the split is in the wrong place (then one half contains the jump and looks very noisy). The sampler tries thousands of
candidate ages and spends time at each one in proportion to how well it explains the record - and that dwell time *is*
the posterior of $T$. The prior on $T$ is flat inside your search window. Ages are in years **b2k** (before 2000 CE),
so *smaller = younger*.

**Step 1.** Choose the event (and optionally your own window and a prior on T).

In [ ]:
EVENT = "Holocene"        # EDIT ME: "Holocene", "Younger Dryas onset" or "Bolling onset"
WINDOW = None             # EDIT ME (optional): (age_min, age_max) in years b2k, to override the preset window
PRIOR_T = None            # EDIT ME (optional): (mean, sd) Normal prior on T in years b2k, e.g. (11700, 50)

**Step 2.** Run the fit (no edits needed).

In [ ]:
if EVENT not in bh.EVENTS:
    raise ValueError(f"EVENT must be one of {list(bh.EVENTS)}.")
ev = bh.EVENTS[EVENT]
if WINDOW is not None and not (isinstance(WINDOW, (tuple, list)) and len(WINDOW) == 2):
    raise ValueError("WINDOW must be None or (age_min, age_max) in years b2k, e.g. (11000, 12600).")
if PRIOR_T is not None and not (isinstance(PRIOR_T, (tuple, list)) and len(PRIOR_T) == 2 and PRIOR_T[1] > 0):
    raise ValueError("PRIOR_T must be None or (mean, sd) in years b2k with sd > 0, e.g. (11700, 50).")
window = tuple(WINDOW) if WINDOW is not None else ev["window"]
if not (9540 <= window[0] < window[1] <= 122280) or window[1] - window[0] < 400:
    raise ValueError("WINDOW must lie inside 9,540-122,280 b2k and span at least 400 years.")

w = ngrip[(ngrip.age_b2k >= window[0]) & (ngrip.age_b2k <= window[1])]
t, yv = w.age_b2k.values, w.d18O_permil.values

log_step, info = bh.step_model(t, yv, window)
if PRIOR_T is not None:
    base = log_step
    mT, sT = PRIOR_T
    log_step = lambda p, base=base, mT=mT, sT=sT: base(p) - 0.5 * ((p[2] - mT) / sT) ** 2

start = [yv[t > np.mean(window)].mean(), yv[t <= np.mean(window)].mean(), np.mean(window), yv.std()]
samples, acc = bh.run_mcmc(log_step, initial=start, step_sizes=[0.08, 0.08, 25, 0.06])
print(f"Acceptance rate: {acc:.0%}")
bh.plot_traces(samples, ["mu_before (‰)", "mu_after (‰)", "T (yr b2k)", "sigma (‰)"], acc)

T = samples[:, 2]
lo, hi = np.percentile(T, [2.5, 97.5])

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax = axes[0]
ax.plot(t, yv, color="0.4", lw=1, label="NGRIP d18O (20-yr means)")
m = samples.mean(axis=0)
tt = np.linspace(window[0], window[1], 500)
ax.plot(tt, np.where(tt > m[2], m[0], m[1]), color="crimson", lw=2, label="posterior-mean step")
ax.axvspan(lo, hi, color="orange", alpha=0.3, label="95% CI of T")
ax.axvline(ev["age"], color="k", ls=":", label=f"reference age: {ev['age']:,} b2k")
ax.invert_xaxis()                                   # younger ages to the right
ax.set_xlabel("Age (years b2k; younger to the right)"); ax.set_ylabel("d18O (‰)")
ax.set_title(f"{EVENT}: the record and the fitted step"); ax.legend(fontsize=8); standardise_label(ax)

ax = axes[1]
ax.hist(T, bins=60, density=True, alpha=0.75)
ax.axvline(ev["age"], color="k", ls=":", label=f"reference age ({ev['age']:,} b2k)")
ax.invert_xaxis()
ax.set_xlabel("Change-point age T (years b2k)"); ax.set_ylabel("Posterior density")
ax.set_title("Posterior of the timing"); ax.legend(fontsize=8); standardise_label(ax)
plt.tight_layout(); plt.show()

print(f"Posterior mean T: {T.mean():,.0f} yr b2k   95% CI [{lo:,.0f}, {hi:,.0f}]  (width {hi - lo:,.0f} yr)")
print(f"Independent reference age: {ev['age']:,} yr b2k (GICC05 ice-core timescale)")
print(f"Size of the step: {m[1] - m[0]:+.2f} ‰ (positive = warming into the younger period)")

**Discuss.**
1. The posterior of T is a fully Bayesian answer: the timing of an event *as a distribution*. What sets its width? Switch to `"Younger Dryas onset"` and `"Bolling onset"`: which transition gives the narrowest posterior, and what in the left panel explains that? (Think: size of the jump compared with the wiggliness of the record.)
2. The search window is itself a prior: "the event is in here somewhere, equally likely everywhere". Make `WINDOW` much wider. Does the posterior care? And what does the posterior look like if the window ends up containing *two* transitions?
3. Now add an informative prior, e.g. `PRIOR_T = (11700, 50)`. What changes in the posterior? Where could such a prior legitimately come from - and when would using one be circular reasoning?
4. Our step is instantaneous, but the real transition took decades, and the data are 20-year means. What is the best timing precision anyone could reach here - and what does the posterior of T *mean* if the truth is a slow ramp rather than a step?

---
## Task 3 - Arctic sea ice: forecast with a prior from climate models

**Goal.** Start from what climate models expect for the decline of September Arctic sea ice (an IPCC/CMIP-based prior),
feed in observations, and watch your belief update. Then test the model on held-out years and make a real forecast.
**Why.** This is Bayesian updating as it is used in practice: the prior comes from physics (models), the data sharpen
it, and the forecast must face reality - the 2020-24 observations, which the fit never saw, and next September.
**You will get** the posterior of the decline rate tightening as years are added, a hold-out test, a 2026 forecast, and
the first ice-free year (area below 1 million km2) as a probability statement.

The model: area = a + b x (year - midpoint)/10 + noise, trained on **1979-TRAIN_END** (the satellite era). b is the rate
in million km2 per decade. The per-year error `tau_sia` is an *assumed* value (0.10 in the satellite era), and the series
is spliced at 2019/2020 - both stated in the data README.

**Step 1.** Choose your knobs. The prior on the rate is given (justify it, test it - that is part of the task):
climate models project roughly 0.6-0.9 million km2 of September ice lost per decade.

In [ ]:
PRIOR_RATE = (-0.75, 0.25)              # EDIT ME: (mean, sd) prior on the rate, mln km2/decade.
                                        #          Try (0.0, 0.25) - "a sceptic: no decline" - or (-0.75, 2.0) - "weak"
TRAIN_END = 2019                        # EDIT ME: last training year (hold out everything after it)
UPDATE_YEARS = [1985, 1995, 2005, 2019] # EDIT ME: checkpoints for the sequential-updating plot (within 1979-TRAIN_END)

**Step 2.** Run the fit, the hold-out test and the forecast (no edits needed).

In [ ]:
if not (1990 <= TRAIN_END <= 2019):
    raise ValueError("TRAIN_END must be between 1990 and 2019 (2020-24 are the splice-adjusted hold-out).")
if not all(1984 <= yy <= TRAIN_END for yy in UPDATE_YEARS):
    raise ValueError(f"UPDATE_YEARS must lie within 1984-{TRAIN_END} (at least 6 training years per fit).")

train = seaice[(seaice.year >= 1979) & (seaice.year <= TRAIN_END)]
hold = seaice[(seaice.year >= 2020) & (seaice.year <= 2024)]

def fit(dsub):
    log_post, info = bh.trend_model(dsub.year.values, dsub.sia_sep_nh.values,
                                    tau=dsub.tau_sia.values, slope_prior=PRIOR_RATE, x0=2000.0)
    s, acc = bh.run_mcmc(log_post, initial=[5.0, PRIOR_RATE[0], 0.5], step_sizes=[0.09, 0.08, 0.06])
    return s, acc, info

# Sequential updating: the posterior of the rate as data accumulate
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax = axes[0]
g = np.linspace(-2.0, 1.0, 400)
ax.plot(g, stats.norm.pdf(g, *PRIOR_RATE), color="k", ls="--", label=f"prior N{PRIOR_RATE} (models)")
for yy in UPDATE_YEARS:
    s_, acc_, _ = fit(train[train.year <= yy])
    ax.hist(s_[:, 1], bins=60, density=True, alpha=0.45, label=f"posterior with data to {yy}")
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Decline rate (mln km² per decade)"); ax.set_ylabel("Density")
ax.set_title("Bayesian updating, year by year"); ax.legend(fontsize=8); standardise_label(ax)

# Full training fit
samples, acc, info = fit(train)
print(f"Acceptance rate (full training fit): {acc:.0%}")
rate = samples[:, 1]

# Posterior predictive: for each sample, the trend line + noise
def predictive(years):
    # trend line + model noise + the (satellite-era) measurement error, variances added
    years = np.asarray(years, float)
    mu = np.array([info["predict"](s, years) for s in samples])
    sd = np.sqrt(samples[:, 2][:, None] ** 2 + 0.10**2)
    return mu + np.random.default_rng(1).normal(0, 1, mu.shape) * sd

years_all = np.arange(1979, 2027)
pp = predictive(years_all)
ax = axes[1]
# note: with TRAIN_END < 2019, the years in between are neither trained on nor tested
ax.errorbar(train.year, train.sia_sep_nh, yerr=2 * train.tau_sia, fmt="o", ms=3, lw=0.8, alpha=0.6, label=f"training 1979-{TRAIN_END}")
ax.errorbar(hold.year, hold.sia_sep_nh, yerr=2 * hold.tau_sia, fmt="s", ms=5, color="crimson", lw=1, label="held-out 2020-24")
ax.plot(years_all, pp.mean(axis=0), color="k", lw=1.5, label="posterior mean")
ax.fill_between(years_all, *np.percentile(pp, [2.5, 97.5], axis=0), color="gray", alpha=0.3, label="95% predictive band")
ax.set_xlabel("Year"); ax.set_ylabel("September sea-ice area (mln km²)")
ax.set_title("Fit, hold-out test and forecast"); ax.legend(fontsize=8); standardise_label(ax)
plt.tight_layout(); plt.show()

lo, hi = np.percentile(rate, [2.5, 97.5])
print(f"Posterior rate:        {rate.mean():.2f} mln km²/decade   95% CI [{lo:.2f}, {hi:.2f}]")
p_hold = predictive(hold.year.values)
inside = ((hold.sia_sep_nh.values >= np.percentile(p_hold, 2.5, axis=0)) &
          (hold.sia_sep_nh.values <= np.percentile(p_hold, 97.5, axis=0)))
print(f"Hold-out 2020-24:      {inside.sum()} of {len(hold)} years inside the 95% predictive band")
f26 = predictive([2026])[:, 0]
print(f"Forecast Sept 2026:    {f26.mean():.2f} mln km²   95% predictive range [{np.percentile(f26, 2.5):.2f}, {np.percentile(f26, 97.5):.2f}]")

# Ice-free year: for each posterior sample, when does the TREND LINE cross 1 mln km2?
a_s, b_s = samples[:, 0], samples[:, 1]
with np.errstate(divide="ignore"):
    year_cross = np.where(b_s < 0, 2000 + 10 * (1.0 - a_s) / b_s, np.inf)   # inf = never (trend not declining)
year_cross = np.where(year_cross > TRAIN_END, year_cross, np.inf)           # a crossing in the past is not physical here
crossing = np.isfinite(year_cross)
p_2070 = np.mean(year_cross < 2070)                 # over ALL samples: a non-declining trend counts as "not before 2070"
if crossing.sum() > 100:
    lo_y, hi_y = np.percentile(year_cross[crossing], [2.5, 97.5])
    print(f"First ice-free September (trend < 1 mln km²): median {np.median(year_cross[crossing]):.0f}, "
          f"95% CI [{lo_y:.0f}, {hi_y:.0f}]  ({crossing.mean():.0%} of samples give a crossing)")
    print(f"P(ice-free before 2070): {p_2070:.2f}   (given this straight-line model; over all posterior samples)")
else:
    print("Too few posterior samples give a declining trend for an ice-free estimate - check your prior.")

**Discuss.**
1. In the left panel: how does the posterior change as years are added? When does it stop looking like the prior? Replace the prior with the sceptic's (`(0.0, 0.25)`): how many years of data does it take to overrule it?
2. Did the hold-out test pass (how many of 2020-24 inside the band)? The last decade of September ice shows a plateau - does your straight line know that? What would failing this test tell you?
3. **Your forecast for 2026.** The notebook printed a prediction for *this* September (2026). The observed September 2026 Arctic sea-ice area, from the [NSIDC Sea Ice Index](https://noaadata.apps.nsidc.org/NOAA/G02135/north/monthly/data/N_09_extent_v4.0.csv), came in at about **3.1 mln km²** (a different satellite product from our series, so treat the comparison as indicative). Does the observation land inside your 95% predictive range - and where in the band? Several recent Septembers sit in the upper half of the band: does that run of high-side years suggest the straight-line decline is too steep? And what can a single year's hit or miss tell you that the five-year hold-out cannot?
4. Your P(ice-free before 2070) comes from extrapolating a straight line for decades. Name two physical reasons the real system might leave the line (in either direction). How much do you trust the probability you printed?

---

*Instructor: Francesco Muschitiello (fm476), October 2026.*
